Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if host == '':
    host = "127.0.0.1"

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376', 'title': 'NORWAY BAG', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/701724376</identifier>\n    <datestamp>2022-04-03T15:00:16.015Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-701724376</identifier><creators><creator><creatorName nameType="Personal">Ola KROGSETH</creatorName><givenNa

In [7]:
record_titles

['NORWAY BAG',
 'Stresses in Mixed Lithium Ion Battery Cathodes',
 'Advanced Photon Source, Sector 21 - The Life Sciences Collaborative Access Team (LS-CAT) - CAT Member Macromolecular Crystallography',
 'IHR',
 'Joint-Structural Biology Group (JSBG) BAG proposal for ID29 beamline',
 'iNEXT Discovery-Instruct ESRF-ERIC; Structural biology for translational research and discovery.',
 'CryoEM BAG for Ile-de-France',
 'Electronic structure of CeO2',
 'Fibers of collagens, secret and unknown of the eardrum: role in the reception of sounds in frogs',
 'Metal uptake and distribution by Toxoplasma gondii',
 'Structural biology at Umeå University',
 'Of bones and teeth: using cross-calibrated histology to infer physiology in mammal ancestors',
 'SAARLAND BAG: ATOMIC MECHANISMS OF ACTION OF MEMBRANE PROTEINS AND BIOSYNTHETIC ENZYMES',
 'NORWAY BAG',
 'Submicron X-ray focusing by single-bounce ellipsoidal capillary combined with parabolic axicon lenses',
 'Time-resolved serial synchrotron crysta

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1413554496</identifier>\n    <datestamp>2024-02-02T07:00:07.996Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1413554496</identifier><creators><creator><creatorName nameType="Personal">Zdzislaw WAWRZAK</creatorName><givenName>Zdzislaw</givenName><familyName>Wawrzak</familyName><nameIdentifier nameIdentifierScheme="ORC

In [9]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-1413554496'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if doi == []:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'MX-2401', 'Particles, Nuclei and Fields', 'BM29', 'Macromolecular Crystallography']


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1221342075', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2097796426', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1450856445', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1976053263', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2290398071', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/542498870', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1851554114', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1317822101', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2156316396', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1304366385', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1409457131', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649420952', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2009044437', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/686521770', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/105931004

In [14]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Cruddos, Joshua', 'name': 'Joshua', 'surname': 'Cruddos', 'rank': 1, 'pid': None}, {'fullName': 'Di Michiel, Marco', 'name': 'Marco', 'surname': 'Di Michiel', 'rank': 2, 'pid': None}, {'fullName': 'Johnson, Michael', 'name': 'Michael', 'surname': 'Johnson', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-5081-4059'}, 'provenance': None}}, {'fullName': 'Llewellyn, Alice', 'name': 'Alice', 'surname': 'Llewellyn', 'rank': 4, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'ID15A'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'IH-MA-375'}, 'provenance': None}], 'mainTitle': 'IHR', 'subTitle': None, 'descriptions': ['IHR'], 'publicationDate': '2026-01-01', 'publisher': None, 'embargoEndDate': None, 'sources': None, 'formats': None, 'contributors': None, 'coverage

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-1221342075', 'subjects': ['ID15A', 'IH-MA-375']},
 {'doi': '10.15151/esrf-es-2290398071',
  'subjects': ['In-house Chemistry', 'ID26', 'IH-CH-1968']},
 {'doi': '10.15151/esrf-es-1450856445',
  'subjects': ['MX-2524', 'ID30A-1', 'Macromolecular Crystallography']},
 {'doi': '10.15151/esrf-es-1273624429',
  'subjects': ['BM07',
   'ID30A-3',
   'Macromolecular Crystallography',
   'XRAYS, X-ray Radiation Technique',
   'PaN',
   'SAD, Single-wavelength Anomalous Diffraction',
   'MX-2599',
   'ID30B',
   'ID23-2',
   'x-ray probe',
   'ID23-1',
   'Particles, Nuclei and Fields',
   'MX-2792',
   'MX-2693',
   'MX, Macromolecular Crystallography',
   'MX-2480']},
 {'doi': '10.15151/esrf-es-1409493638',
  'subjects': ['BM07',
   'ID30A-3',
   'Macromolecular Crystallography',
   'XRAYS, X-ray Radiation Technique',
   'PaN',
   'SAD, Single-wavelength Anomalous Diffraction',
   'MX-2599',
   'ID30B',
   'ID23-2',
   'x-ray probe',
   'ID23-1',
   'Particles, Nuclei

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376',
  'doi': '10.15151/esrf-es-701724376'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780',
  'doi': '10.15151/esrf-es-969290780'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496',
  'doi': '10.15151/esrf-es-1413554496'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1221342075',
  'doi': '10.15151/esrf-es-1221342075'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2097796426',
  'doi': '10.15151/esrf-es-2097796426'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1450856445',
  'doi': '10.15151/esrf-es-1450856445'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1976053263',
  'doi': '10.15151/esrf-es-1976053263'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2290398071',
  'doi': '10.15151/esrf-es-2290398071'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/542498870',
  'doi': '10.15151/esrf-es-542498870'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1851554114

In [18]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376',
  'doi': '10.15151/esrf-es-701724376',
  'enriched_subjects': {'BM29',
   'MX-2401',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780',
  'doi': '10.15151/esrf-es-969290780',
  'enriched_subjects': {'7. Clean energy',
   'Applied Material Science',
   'BM32',
   'MA-5392',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496',
  'doi': '10.15151/esrf-es-1413554496',
  'enriched_subjects': {'BM07',
   'ID23-1',
   'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX, Macromolecular Crystallography',
   'MX-2480',
   'MX-2599',
   'MX-2693',
   'MX-2792',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'SAD, Single-wavelength Anomalous Diffraction',
   'XRAYS, X-ray Radiation Technique',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424d

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376',
  'doi': '10.15151/esrf-es-701724376',
  'enriched_subjects': {'BM29',
   'MX-2401',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780',
  'doi': '10.15151/esrf-es-969290780',
  'enriched_subjects': {'7. Clean energy',
   'Applied Material Science',
   'BM32',
   'MA-5392',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496',
  'doi': '10.15151/esrf-es-1413554496',
  'enriched_subjects': {'BM07',
   'ID23-1',
   'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX, Macromolecular Crystallography',
   'MX-2480',
   'MX-2599',
   'MX-2693',
   'MX-2792',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'SAD, Single-wavelength Anomalous Diffraction',
   'XRAYS, X-ray Radiation Technique',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424d

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376',
  'doi': '10.15151/esrf-es-701724376',
  'enriched_subjects': {'BM29',
   'MX-2401',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780',
  'doi': '10.15151/esrf-es-969290780',
  'enriched_subjects': {'7. Clean energy',
   'Applied Material Science',
   'BM32',
   'MA-5392',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496',
  'doi': '10.15151/esrf-es-1413554496',
  'enriched_subjects': {'BM07',
   'ID23-1',
   'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX, Macromolecular Crystallography',
   'MX-2480',
   'MX-2599',
   'MX-2693',
   'MX-2792',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'SAD, Single-wavelength Anomalous Diffraction',
   'XRAYS, X-ray Radiation Technique',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424d

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

150
150


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [24]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701724376',
  'doi': '10.15151/esrf-es-701724376',
  'enriched_subjects': {'BM29',
   'MX-2401',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/969290780',
  'doi': '10.15151/esrf-es-969290780',
  'enriched_subjects': {'7. Clean energy',
   'Applied Material Science',
   'BM32',
   'MA-5392',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1413554496',
  'doi': '10.15151/esrf-es-1413554496',
  'enriched_subjects': {'BM07',
   'ID23-1',
   'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX, Macromolecular Crystallography',
   'MX-2480',
   'MX-2599',
   'MX-2693',
   'MX-2792',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'SAD, Single-wavelength Anomalous Diffraction',
   'XRAYS, X-ray Radiation Technique',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424d

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/701724376</identifier>\n    <datestamp>2022-04-03T15:00:16.015Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-701724376</identifier><creators><creator><creatorName nameType="Personal">Ola KROGSETH</creatorName><givenName>Ola</givenName><familyName>Krogseth</familyName></creator><creator><creatorName nameType="Personal

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [30]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
data

{'id': 'https://openalex.org/W6886229830',
 'doi': 'https://doi.org/10.15151/esrf-es-1473312981',
 'title': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'display_name': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'publication_year': 2024,
 'publication_date': '2024-01-25',
 'ids': {'openalex': 'https://openalex.org/W6886229830',
  'doi': 'https://doi.org/10.15151/esrf-es-1473312981'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.15151/esrf-es-1473312981',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.15151/esrf-es-1473312981',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S7407051764',
   'display_name': 'European Synchrotron Radiation Facility',
   'issn_l': None,
   'issn': [],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'host_organization': None,
   'host_organiza

In [ ]:
# Get potential subjects, two relevant fields: keywords and concepts

In [ ]:
def get_works(filter_str, per_page=100):
    """Fetch works with pagination."""
    url = "https://api.openalex.org/works"
    params = {
        "filter": filter_str,
        "per_page": per_page,
        "cursor": "*",
        "api_key": "YOUR_KEY"
    }

    all_works = []
    while True:
        response = requests.get(url, params=params).json()
        all_works.extend(response["results"])

        cursor = response["meta"].get("next_cursor")
        if not cursor:
            break
        params["cursor"] = cursor

    return all_works

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword